In [ ]:
import pandas as pd
import numpy as np
import pickle



In [ ]:
# import csv of imagej analyzed data
df = pd.read_csv("/content/drive/Shareddrives/Summer_2023_VCell_Ml/Summer 2023  Image analysis/param_scan_7-6-23/=7-6_analysis_CSV.csv")
df.head()

,File_Name,avg_val,num_spots,total_coverage,avg_spot_size,spot_size_std,avg_circ,circ_std
0,"Ba=-0.038,Da=0.0,Di=0.5,Ga=0.075",79.2727,724.0,0.326225,1.126500e-08,0.000523,0.713834,0.291812
1,"Ba=-0.038,Da=0.0,Di=0.5,Ga=0.1425",224.3437,1.0,0.923225,2.308062e-05,0.000000,0.541700,0.000000
2,"Ba=-0.038,Da=0.0,Di=0.5,Ga=0.21",241.8883,1.0,0.995425,2.488562e-05,0.000000,0.760000,0.000000
3,"Ba=-0.038,Da=0.0,Di=0.5,Gi=0.0227",239.9686,1.0,0.987525,2.468813e-05,0.000000,0.739200,0.000000
4,"Ba=-0.038,Da=0.0,Di=0.5,Gi=0.07135",135.4482,74.0,0.557400,1.883110e-07,0.060858,0.842827,0.243896


In [ ]:

# indexed as[wt_value,min,middle,max]
Param_Vals = {"Ua":[0.03,0.0083,0.02265,0.037],"Ui":[0.07,0.045,0.1425,0.24],
          "Ga":[0.08, 0.075,0.1425,0.21],"Gi":[0.1,0.0227,0.07135,0.12],
          "Ba":[-0.12,-0.138,-0.088,-0.038],"Da":[0.01,0,0.0065,0.013],
          "Di":[0.5,1.75,3]}

Param_List = {"Ua": [], "Ui": [], "Ga": [], "Gi" :[], "Ba": [], "Da": [], "Di": []}

for name in df["File_Name"]: #iterate through al file names
  for param in Param_Vals.keys(): #iterate through all parameter names
    if param in name: #check if parameter name is in file name
      p_i = name.index(param) #get parameter name index
      k = p_i + 3 #index of first number value of parameter
      while k < len(name) and name[k] != ",": #iterate through file name until hitting a comma or the end
        k += 1
      Param_List[param].append(name[p_i+3:k]) #add that value to the parameter list index for that parameter
    else:
      Param_List[param].append(Param_Vals[param][0]) #if the parameter is not in the file name, add the wild-type value to the parameter list


In [ ]:
j = 1
#adds columns for all parameters to the dataframe, can only run once per session
for i in Param_List.keys():
  df.insert(j, i, Param_List[i])
  j+=1
df

,File_Name,Ua,Ui,Ga,Gi,Ba,Da,Di,avg_val,num_spots,total_coverage,avg_spot_size,spot_size_std,avg_circ,circ_std
0,"Ba=-0.038,Da=0.0,Di=0.5,Ga=0.075",0.03,0.07,0.075,0.1,-0.038,0.0,0.5,79.2727,724.0,0.326225,1.126500e-08,0.000523,0.713834,0.291812
1,"Ba=-0.038,Da=0.0,Di=0.5,Ga=0.1425",0.03,0.07,0.1425,0.1,-0.038,0.0,0.5,224.3437,1.0,0.923225,2.308062e-05,0.000000,0.541700,0.000000
2,"Ba=-0.038,Da=0.0,Di=0.5,Ga=0.21",0.03,0.07,0.21,0.1,-0.038,0.0,0.5,241.8883,1.0,0.995425,2.488562e-05,0.000000,0.760000,0.000000
3,"Ba=-0.038,Da=0.0,Di=0.5,Gi=0.0227",0.03,0.07,0.08,0.0227,-0.038,0.0,0.5,239.9686,1.0,0.987525,2.468813e-05,0.000000,0.739200,0.000000
4,"Ba=-0.038,Da=0.0,Di=0.5,Gi=0.07135",0.03,0.07,0.08,0.07135,-0.038,0.0,0.5,135.4482,74.0,0.557400,1.883110e-07,0.060858,0.842827,0.243896
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2830,"Ga=0.21,Gi=0.12,Ua=0.02265,Ui=0.1425",0.02265,0.1425,0.21,0.12,-0.12,0.01,0.5,255.0000,1.0,1.000000,2.500000e-05,0.000000,0.790000,0.000000
2831,"Ga=0.21,Gi=0.12,Ua=0.02265,Ui=0.24",0.02265,0.24,0.21,0.12,-0.12,0.01,0.5,255.0000,1.0,1.000000,2.500000e-05,0.000000,0.790000,0.000000
2832,"Ga=0.21,Gi=0.12,Ua=0.037,Ui=0.045",0.037,0.045,0.21,0.12,-0.12,0.01,0.5,26.2136,73.0,0.190450,6.522300e-08,0.000883,0.954325,0.082575
2833,"Ga=0.21,Gi=0.12,Ua=0.037,Ui=0.1425",0.037,0.1425,0.21,0.12,-0.12,0.01,0.5,197.0000,1.0,1.000000,2.500000e-05,0.000000,0.790000,0.000000


In [ ]:
df = df.drop(columns=["File_Name"]) #get rrif of File_Name column
df


In [ ]:
df = df.astype(float) #change all values to floats

In [ ]:
#load PCA image features saved as pickle file
with open('/content/drive/Shareddrives/Summer_2023_VCell_Ml/Summer 2023  Image analysis/param_scan_7-6-23/PCA_Data_Pickle_7-20.pkl', 'rb') as f:

    data = pickle.load(f) # deserialize using load()
    print(data)


Building the model

Adapted from

In [ ]:
#create train test split and divide dataset into inputs and targets
from sklearn.model_selection import train_test_split
x = data
y = df[["Ua","Ui","Ga","Gi","Ba","Da","Di"]]
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size = 0.2)


Random Forest Model

In [ ]:
# Import the model we are using
from sklearn.ensemble import RandomForestRegressor
# Instantiate model with 1000 decision trees
rf = RandomForestRegressor(n_estimators = 1000, random_state = 42)
# Train the model on training data
rf.fit(x_train, y_train);

In [ ]:
y_test

,Ua,Ui,Ga,Gi,Ba,Da,Di
313,0.03000,0.1425,0.0800,0.12000,-0.038,0.0100,0.50
2604,0.00830,0.0700,0.2100,0.12000,-0.120,0.0100,1.75
2004,0.03000,0.0450,0.0750,0.10000,-0.120,0.0065,3.00
825,0.03000,0.0450,0.1425,0.10000,-0.088,0.0100,0.50
1757,0.03700,0.0700,0.0800,0.02270,-0.120,0.0000,3.00
...,...,...,...,...,...,...,...
766,0.03000,0.0700,0.1425,0.07135,-0.088,0.0130,0.50
127,0.03000,0.0700,0.0750,0.07135,-0.038,0.0065,0.50
1210,0.02265,0.0700,0.0750,0.10000,-0.138,0.0065,0.50
1922,0.03000,0.2400,0.0800,0.02270,-0.120,0.0065,0.50


In [ ]:
# Use the forest's predict method on the test data
predictions = rf.predict(x_test)
# Calculate the absolute errors
errors = abs(predictions - y_test)
# Print out the mean absolute error (mae)
# print('Mean Absolute Error:')
# print(round(np.mean(errors), 5))

# Calculate mean absolute percentage error (MAPE)
mape = 100 * (errors / abs(y_test))
print("CNN Features")
print("Mean Absolute Percent Error")
print(round(np.mean(mape),5), "%")
# Calculate and display accuracy
# accuracy = 100 - np.mean(mape)
# print('Accuracy:', round(accuracy, 5), '%.')

CNN Features
Mean Absolute Percent Error
Ua    28.74080
Ui    41.78195
Ga    27.06410
Gi    52.15157
Ba    40.08022
Da         inf
Di    67.70590
dtype: float64 %


/usr/local/lib/python3.10/dist-packages/numpy/core/fromnumeric.py:3472: FutureWarning: In a future version, DataFrame.mean(axis=None) will return a scalar mean over the entire DataFrame. To retain the old behavior, use 'frame.mean(axis=0)' or just 'frame.mean()'
  return mean(axis=axis, dtype=dtype, out=out, **kwargs)


In [ ]:
#WT Prediction
wt_df = pd.DataFrame({"avg_val":[7.1611] ,"num_spots":[83.0] ,"total_coverage":[0.056249999999999994],
                      "avg_spot_size":[float(1.6942771084337348e-08)] ,"spot_size_std": [0.00011525326016563174],"avg_circ":[0.9659421686746988] ,"circ_std":[0.04273925742706969]})


wt_prediction = rf.predict(wt_df)
wt_prediction

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:432: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(


ValueError: ignored